# Datamine INPFML Process Example

This notebook demonstrates how to configure and run the **`inpfml`** process wrapper in `dmstudio`.

## Process Description

## INPFML

Creates a file and enters data of fixed format into it. The data may come either from the keyboard, or from a system file external to the database; the system file records must be no longer than 80 characters.

The data format may come either from the file description, entered when the Data Definition is requested, or from the format line.

1\.  |  INPFML may be considered as a combination of [INPUTD](<inputd.md>). Incorrect data records (i.e. not matching the specified format statement) will be ignored, with a message to the display.
---|---
|  |
2. |  Records may not start with !. This is because the ! symbol acts as the end-of-data character.
|  |
3. |  Macro files, in which each command starts with !, are best entered through [INPUTC](<inputc.md>).
|  |
4. |  Within a macro, the Data Definition cannot be terminated by the ! character, as this will terminate the **INPFML** process; instead another special purpose character (such as [) may be used.
|  |
5. |  The user is prompted for the following information
|  >FILE DESCRIPTION > |  Up to 60 character text file description, or a FORTRAN format in brackets, defining the format of data to be read in (e.g. by the INPUTF or INPUTW processes). Order of fields is that in the DD and the type of each field must match that in the DD. Character data must be in A4 units (e.g. 3A4,A2), numbers in E, F or G format (e.g. 3F12.0), integers as F format (e.g. F6.0) and X for spaces. Maximum length of format statement is 80 characters, including the enclosing brackets, and maximum length of data record to which format applies is also 80 characters.
|  > NEXT FIELDNAME > |  Up to 24 character field 'name'. May be upper or lower case text, numbers, or special symbols. Internal blanks in field names should be avoided, and _ / . - used instead. Fields should not start with !, *, & or @, as such names will give rise to syntax problems under some circumstances.
|  > TYPE A/N (N) > |  A for alphanumeric field, N or return for numeric field.
|  > LENGTH (1) > |  Length in characters for alphanumeric fields only. Question omitted for numeric fields.
|  > STORED? (Y) > |  Y or return for explicit fields, N for implicit (file constant) fields. Implicit fields are not supplied in data records.
|  > DEFAULT > |  Default value to replace absent data (in processes which carry out this replacement). Just <return> for blank default value for alphanumeric and zero for numeric fields. For implicit fields the constant values must be entered.
|  The above is repeated for each field until ! entered for NEXT FIELDNAME. Then:
|  > CONFIRM? > |  OK, ok, YES, Y or y to end DD.
|  > SYSFILE > |  External system file name from which the formatted data is to be read, or return for entry from the keyboard. Format of the system file name is system specific, with a maximum of 56 characters allowed, including pathnames.
|  > FORMAT > |  FORTRAN format in brackets, defining the format of data records to be read. Order of fields is that in the DD and the type of each field must match that in the DD. Character data must be in A4 units (e.g. 3A4,A2), numbers in E, F or G format (e.g. 3F12.0), integers as F format (e.g. F6.0) and X for spaces. Maximum length of format statement is 80 characters, including the enclosing brackets, and maximum length of data record to which format applies is also 80 characters. If format is already in the FILE DESCRIPTION, then enter just return.
|  > DATA > |  If no SYSFILE, data lines in defined format. ! terminates.

### Input Files:

### Output Files:

* **out** (Table):
  File to be created.
  Required=Yes

### Fields:

### Parameters:

* **print**:
  >=1 to display each record (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('inpfml')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute inpfml
print("Running inpfml...")
dm_fil.inpfml(
    out_o='t_inpfml_out',  # required
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("inpfml execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_inpfml_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")